In [1]:
# ==================================================
# LOAD DOB SILVER
# ==================================================

dob_df = spark.read.parquet(
    minio_path("silver/dob")
)

print(
    "DOB rows:",
    dob_df.count()
)

print("\nDOB SILVER SCHEMA")

dob_df.printSchema()

NameError: name 'spark' is not defined

In [2]:
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# ==================================================
# PROJECT CONFIG
# ==================================================

PROJECT_ROOT = "/workspace/nyc-building-risk"
COMMON_PATH = f"{PROJECT_ROOT}/spark/common"

if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)


# ==================================================
# IMPORT PROJECT HELPERS
# ==================================================

from minio_config import configure_minio, minio_path


# ==================================================
# CREATE / GET SPARK SESSION
# ==================================================

spark = (
    SparkSession.builder
    .appName("NYC Building Risk - Gold DOB Violations")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .getOrCreate()
)


# ==================================================
# CONFIGURE MINIO / S3A
# ==================================================

configure_minio(spark)


# ==================================================
# REDUCE LOG NOISE
# ==================================================

spark.sparkContext.setLogLevel("WARN")


# ==================================================
# CONNECTION TEST
# ==================================================

print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print("MinIO helper loaded successfully")
print("Test:", spark.range(1).count())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/07 18:05:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/07 18:05:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/07 18:05:59 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/09/07 18:05:59 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/09/07 18:05:59 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


Spark version: 3.4.0
Master: local[2]
MinIO helper loaded successfully
Test: 1


In [3]:
# ==================================================
# LOAD DOB SILVER
# ==================================================

dob_df = spark.read.parquet(
    minio_path("silver/dob")
)

print(
    "DOB rows:",
    dob_df.count()
)

print("\nDOB SILVER SCHEMA")

dob_df.printSchema()


26/09/07 18:06:16 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


DOB rows: 148688

DOB SILVER SCHEMA
root
 |-- violation_number: string (nullable = true)
 |-- violation_issue_date: timestamp (nullable = true)
 |-- violation_status: string (nullable = true)
 |-- violation_type: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- bbl: string (nullable = true)
 |-- bin: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- block: string (nullable = true)
 |-- lot: string (nullable = true)
 |-- house_number: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- community_board: string (nullable = true)
 |-- council_district: string (nullable = true)
 |-- census_tract_2020_: string (nullable = true)
 |-- neighborhood_tabulation_area_nta_2020_: string (nullable = true)
 |-- source_bbl: string (nullable = true)
 |-- bbl_s

In [4]:
# ==================================================
# DOB BASIC VALIDATION
# ==================================================

print(
    "Total DOB rows:",
    dob_df.count()
)

print(
    "Distinct violation_number:",
    dob_df
    .select("violation_number")
    .distinct()
    .count()
)

print(
    "Missing violation_number:",
    dob_df
    .filter(F.col("violation_number").isNull())
    .count()
)

print(
    "With BIN:",
    dob_df
    .filter(
        F.col("bin").isNotNull()
        & (F.trim(F.col("bin")) != "")
    )
    .count()
)

print(
    "Without BIN:",
    dob_df
    .filter(
        F.col("bin").isNull()
        | (F.trim(F.col("bin")) == "")
    )
    .count()
)

Total DOB rows: 148688


Distinct violation_number: 148688


Missing violation_number: 0


With BIN: 148688


Without BIN: 0


In [5]:
# ==================================================
# LOAD DIM_BUILDING
# ==================================================

dim_building_df = spark.read.parquet(
    minio_path(
        "gold/data_model/dim_building"
    )
)

print(
    "dim_building rows:",
    dim_building_df.count()
)


# ==================================================
# DOB BIN -> DIM_BUILDING
# ==================================================

building_lookup = (
    dim_building_df
    .select(
        F.col("bin")
        .cast("string")
        .alias("source_bin"),

        "building_id",

        F.col("property_id")
        .alias("building_property_id")
    )
    .dropDuplicates(
        ["source_bin"]
    )
)


dob_stage = (
    dob_df

    .withColumn(
        "source_bin",
        F.col("bin").cast("string")
    )

    .withColumn(
        "source_bbl_original",
        F.col("bbl").cast("string")
    )

    .join(
        building_lookup,
        on="source_bin",
        how="left"
    )
)

dim_building rows: 197958


In [6]:
print(
    "DOB rows:",
    dob_stage.count()
)

print(
    "With building_id:",
    dob_stage
    .filter(F.col("building_id").isNotNull())
    .count()
)

print(
    "Without building_id:",
    dob_stage
    .filter(F.col("building_id").isNull())
    .count()
)

print(
    "With canonical property_id:",
    dob_stage
    .filter(F.col("building_property_id").isNotNull())
    .count()
)

print(
    "Without canonical property_id:",
    dob_stage
    .filter(F.col("building_property_id").isNull())
    .count()
)

DOB rows: 148688


With building_id: 148688


Without building_id: 0


With canonical property_id: 147894


Without canonical property_id: 794


In [7]:
# ==================================================
# LOAD DIM_PROPERTY
# ==================================================

dim_property_df = spark.read.parquet(
    minio_path(
        "gold/data_model/dim_property"
    )
)


# ==================================================
# DOB SOURCE BBL -> DIM_PROPERTY
# ==================================================

property_lookup = (
    dim_property_df
    .select(
        F.col("bbl")
        .cast("string")
        .alias("source_bbl_original"),

        F.col("property_id")
        .alias("source_property_id")
    )
    .dropDuplicates(
        ["source_bbl_original"]
    )
)


dob_stage_v2 = (
    dob_stage
    .join(
        property_lookup,
        on="source_bbl_original",
        how="left"
    )
)

In [8]:
both_property = (
    F.col("building_property_id").isNotNull()
    & F.col("source_property_id").isNotNull()
)

print(
    "Both Property IDs available:",
    dob_stage_v2
    .filter(both_property)
    .count()
)

print(
    "Same Property:",
    dob_stage_v2
    .filter(
        both_property
        & (
            F.col("building_property_id")
            == F.col("source_property_id")
        )
    )
    .count()
)

print(
    "Different Property:",
    dob_stage_v2
    .filter(
        both_property
        & (
            F.col("building_property_id")
            != F.col("source_property_id")
        )
    )
    .count()
)

print(
    "Building Property only:",
    dob_stage_v2
    .filter(
        F.col("building_property_id").isNotNull()
        & F.col("source_property_id").isNull()
    )
    .count()
)

print(
    "Source Property only:",
    dob_stage_v2
    .filter(
        F.col("building_property_id").isNull()
        & F.col("source_property_id").isNotNull()
    )
    .count()
)

print(
    "No Property:",
    dob_stage_v2
    .filter(
        F.col("building_property_id").isNull()
        & F.col("source_property_id").isNull()
    )
    .count()
)

Both Property IDs available: 147422


Same Property: 147377


Different Property: 45


Building Property only: 472


Source Property only: 0


No Property: 794


In [9]:
# ==================================================
# FINAL PROPERTY RESOLUTION FOR DOB
# ==================================================

dob_final_stage = (
    dob_stage_v2

    .withColumn(
        "property_id",
        F.when(
            F.col("building_property_id").isNotNull(),
            F.col("building_property_id")
        )
        .when(
            F.col("building_id").isNull()
            & F.col("source_property_id").isNotNull(),
            F.col("source_property_id")
        )
    )

    .withColumn(
        "property_resolution_method",
        F.when(
            F.col("building_property_id").isNotNull(),
            F.lit("BUILDING_CANONICAL")
        )
        .when(
            F.col("building_id").isNull()
            & F.col("source_property_id").isNotNull(),
            F.lit("SOURCE_BBL")
        )
        .otherwise(
            F.lit("UNRESOLVED")
        )
    )

    .withColumn(
        "property_conflict_flag",
        F.when(
            F.col("building_property_id").isNotNull()
            & F.col("source_property_id").isNotNull()
            & (
                F.col("building_property_id")
                != F.col("source_property_id")
            ),
            F.lit(1)
        ).otherwise(F.lit(0))
    )

    .withColumn(
        "building_resolution_status",
        F.when(
            F.col("building_id").isNotNull(),
            F.lit("RESOLVED")
        ).otherwise(
            F.lit("UNRESOLVED")
        )
    )
)

In [10]:
print(
    "Total DOB rows:",
    dob_final_stage.count()
)

print(
    "With building_id:",
    dob_final_stage
    .filter(F.col("building_id").isNotNull())
    .count()
)

print(
    "With property_id:",
    dob_final_stage
    .filter(F.col("property_id").isNotNull())
    .count()
)

print(
    "Without property_id:",
    dob_final_stage
    .filter(F.col("property_id").isNull())
    .count()
)

print(
    "Property conflicts:",
    dob_final_stage
    .filter(F.col("property_conflict_flag") == 1)
    .count()
)

(
    dob_final_stage
    .groupBy("property_resolution_method")
    .count()
    .show(truncate=False)
)

Total DOB rows: 148688


With building_id: 148688


With property_id: 147894


Without property_id: 794


Property conflicts: 45


+--------------------------+------+
|property_resolution_method|count |
+--------------------------+------+
|UNRESOLVED                |794   |
|BUILDING_CANONICAL        |147894|
+--------------------------+------+



In [11]:
# ==================================================
# FINAL FACT_DOB_VIOLATION
# Grain: 1 row = 1 DOB violation
# ==================================================

fact_dob_violation = (
    dob_final_stage

    .withColumn(
        "dob_event_id",
        F.concat(
            F.lit("DOB:"),
            F.col("violation_number")
        )
    )

    .select(
        # Keys
        "dob_event_id",
        "violation_number",

        "property_id",
        "building_id",

        # Identity / Audit
        "source_bin",
        "source_bbl_original",
        "source_property_id",
        "building_property_id",

        "property_resolution_method",
        "property_conflict_flag",
        "building_resolution_status",

        # Violation
        "violation_issue_date",
        "violation_status",
        "violation_type",
        "device_type",

        # Address
        "borough",
        "block",
        "lot",
        "house_number",
        "street",
        "city",
        "state",
        "zip",

        "latitude",
        "longitude",

        "community_board",
        "council_district",
        "census_tract_2020_",
        "neighborhood_tabulation_area_nta_2020_",

        # Original source metadata
        "source_bbl",
        "bbl_source",

        # Calendar helpers
        "violation_day",
        "violation_year",
        "violation_month"
    )
)

In [12]:
print(
    "Fact rows:",
    fact_dob_violation.count()
)

print(
    "Distinct dob_event_id:",
    fact_dob_violation
    .select("dob_event_id")
    .distinct()
    .count()
)

Fact rows: 148688


Distinct dob_event_id: 148688


In [13]:
# ==================================================
# SAVE FACT_DOB_VIOLATION
# ==================================================

FACT_DOB_PATH = minio_path(
    "gold/data_model/fact_dob_violation"
)

(
    fact_dob_violation
    .write
    .mode("overwrite")
    .parquet(FACT_DOB_PATH)
)

print("fact_dob_violation saved successfully")
print("Path:", FACT_DOB_PATH)

26/09/07 18:11:57 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


fact_dob_violation saved successfully
Path: s3a://nyc-building-risk/gold/data_model/fact_dob_violation
